In [3]:
import warnings
warnings.filterwarnings("ignore")

import sys, os

# Add the parent directory (project_root) to sys.path
PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.append(PARENT_DIR)

from gt_concepts.nucleolus import exact_nucleolus


import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


# Utility wrapper for nucleolus game (regression, same v(S) as least-core)

class _FeatureDataWrapper:
    """Wrapper so exact_nucleolus can query feature names / number of players."""
    def __init__(self, feature_names):
        self.data_names = list(feature_names)

    def __len__(self):
        return len(self.data_names)


class RegressionFeatureUtility:
    """
    Cooperative game for regression:
        - Players = features (columns of X)
        - Coalition S (list of indices) = subset of features
        - v(S) = average predicted target when only features in S
                are kept from X_test and others are set to a baseline.
                (Identical v(S) to your least-core regression wrapper.)
    """
    def __init__(self, model, X_train, X_test, feature_names):
        self.model = model
        self.X_train = X_train.reset_index(drop=True)
        self.X_test = X_test.reset_index(drop=True)
        self.feature_names = list(feature_names)
        self.data = _FeatureDataWrapper(self.feature_names)

        # Baseline value for missing features (mean over training data)
        self.baseline = self.X_train.mean(axis=0).values

    def __call__(self, coalition):
        coalition = list(coalition)

        if not coalition:
            # empty coalition = no predictive power
            return 0.0

        # Start from baseline matrix
        X_masked = np.tile(self.baseline, (self.X_test.shape[0], 1))

        # Put back the "active" coalition features from X_test
        X_masked[:, coalition] = self.X_test.iloc[:, coalition].values

        # Predict and take mean prediction as coalition value
        preds = self.model.predict(X_masked)
        return float(preds.mean())


# Nucleolus attribution for regression

def compute_nucleolus_attribution_reg(model, X_train, X_test, feature_names, solver=None):
    """
    Returns: pd.Series of nucleolus attributions for a regression model,
    using the same v(S) as your least-core implementation.
    """
    utility = RegressionFeatureUtility(model, X_train, X_test, feature_names)

    options = {}
    if solver:
        options["solver"] = solver  # e.g. "ECOS", "SCS"

    result = exact_nucleolus(utility, options=options)
    return pd.Series(result.values, index=feature_names, name="Nucleolus")


def load_regression_csv(path, target_col=None, test_size=0.25, random_state=42):
    """
    Load a CSV, drop 'Unnamed' index columns, split into train/test.
    If target_col is None, use the last column as target.
    """
    df = pd.read_csv(path)

    # Drop index-like columns such as 'Unnamed: 0'
    df = df.loc[:, ~df.columns.str.contains(r'^Unnamed')]

    if target_col is None:
        target_col = df.columns[-1]

    y = df[target_col].values
    X = df.drop(columns=[target_col])

    feature_names = list(X.columns)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    return X_train, X_test, y_train, y_test, feature_names


def run_nucleolus_on_csv(path, target_col, dataset_name):
    print("\n" + "=" * 72)
    print(f"Nucleolus (regression) on: {dataset_name}")
    print("=" * 72)

    X_train, X_test, y_train, y_test, feature_names = load_regression_csv(
        path, target_col=target_col
    )

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=42,
    )
    model.fit(X_train, y_train)

    # Compute nucleolus attributions
    nucleolus_values = compute_nucleolus_attribution_reg(
        model,
        X_train,
        X_test,
        feature_names,
        solver=None,
    )

    print("\n===== Nucleolus Attributions =====")
    print(nucleolus_values.sort_values(ascending=False))
    return nucleolus_values


if __name__ == "__main__":
    LUNG_PATH = "../datasets/lung_cancer_risk.csv"
    CV_PATH   = "../datasets/cv_risk.csv"

    # Lung cancer: target column is 'lung_cancer_risk'
    lung_nucleolus = run_nucleolus_on_csv(
        LUNG_PATH,
        target_col="lung_cancer_risk",
        dataset_name="synthetic_lung_cancer_risk",
    )

    # Cardiovascular risk: target column is 'cv_risk'
    cv_nucleolus = run_nucleolus_on_csv(
        CV_PATH,
        target_col="cv_risk",
        dataset_name="synthetic_cardiovascular_risk",
    )


Nucleolus (regression) on: synthetic_lung_cancer_risk

===== Nucleolus Attributions =====
stress          5.631158
drink_coffee    5.386978
smoking         4.969614
Name: Nucleolus, dtype: float64

Nucleolus (regression) on: synthetic_cardiovascular_risk

===== Nucleolus Attributions =====
family_history    3.108818
diet_score        2.885554
sleep_duration    2.681135
bmi               2.497536
Name: Nucleolus, dtype: float64
